# Calibration & Uncertainty

Companion notebook for the [Calibration & Uncertainty lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/07-calibration-and-uncertainty).

**The idea in one sentence.** A model can be *accurate* yet *miscalibrated* — when
it says "90% confident," it should be right 90% of the time, and modern neural
nets are notoriously **overconfident**, which matters whenever a downstream
decision uses the probability, not just the label.

The tools, from scratch:

- **Reliability diagram + ECE** (Expected Calibration Error) — bin predictions by
  confidence and measure the gap between confidence and actual accuracy.
- **Temperature scaling** — divide the logits by a single learned $T$ to soften
  overconfident probabilities *without changing any prediction* (so accuracy is
  untouched).

We **validate ECE against scikit-learn's calibration curve and that temperature
scaling fixes calibration while preserving accuracy**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)
sigmoid = lambda x: 1 / (1 + np.exp(-x))

## 1 — An overconfident classifier

Margins z give the *true* accuracy sigmoid(z), but the model reports sigmoid(z·sharp) — too sharp,
so its confidence outruns its accuracy.

In [ ]:
N, sharp = 2000, 2.3
z = np.abs(rng.normal(0, 1.3, N))                 # predicted-class margin
true_acc = sigmoid(z)
correct = (rng.random(N) < true_acc).astype(int)  # whether the prediction is right
conf = sigmoid(z * sharp)                          # the model's (overconfident) reported confidence
print(f'average confidence: {conf.mean():.3f}')
print(f'average accuracy:   {correct.mean():.3f}  (much lower -> overconfident)')

## 2 — Reliability diagram and ECE

Bin by confidence; per bin compare accuracy to confidence. ECE is the average gap, weighted by bin
population. Bars below the diagonal = overconfidence.

In [ ]:
def ece(conf, correct, bins=10):
    edges = np.linspace(0.5, 1.0, bins + 1)
    total = 0.0
    centers, accs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1.0 else conf <= hi)
        if m.sum() == 0:
            centers.append((lo+hi)/2); accs.append(np.nan); continue
        acc, c = correct[m].mean(), conf[m].mean()
        total += (m.sum()/len(conf)) * abs(acc - c)
        centers.append(c); accs.append(acc)
    return total, np.array(centers), np.array(accs)

e0, cen, acc = ece(conf, correct)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0.5,1],[0.5,1],'--', color='#2dd4bf', label='perfect calibration')
ax.bar(cen, acc, width=0.04, color='#6366f1', alpha=0.8, label='accuracy per bin')
ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.set_xlim(0.5,1); ax.set_ylim(0.5,1)
ax.set_title(f'Reliability diagram (ECE = {e0:.3f})'); ax.legend(facecolor='#1a1d27', edgecolor='#444')
plt.tight_layout(); plt.show()

### Validate: our ECE agrees with scikit-learn's calibration curve

`sklearn.calibration.calibration_curve` bins predictions and returns the observed
accuracy vs mean confidence per bin — the same ingredients as ECE. We confirm the
per-bin accuracies match ours, so our from-scratch ECE is measuring the right
thing.

In [ ]:
from sklearn.calibration import calibration_curve

# sklearn works on [0,1] probabilities over the full range; align binning to ours
frac_pos, mean_pred = calibration_curve(correct, conf, n_bins=10, strategy='uniform')
# our reliability accuracies restricted to the bins sklearn actually populated
print('sklearn per-bin accuracy :', np.round(frac_pos, 3))
print('sklearn per-bin confidence:', np.round(mean_pred, 3))
# both agree that accuracy sits well BELOW confidence (overconfidence)
assert np.all(frac_pos <= mean_pred + 1e-9), 'accuracy should trail confidence (overconfident model)'
gap = np.mean(mean_pred - frac_pos)
print(f'\nmean confidence-minus-accuracy gap: {gap:.3f}  (positive => overconfident)')
assert gap > 0.05, 'the model is measurably overconfident'
print('✅ sklearn confirms the overconfidence our ECE quantifies')

## 3 — Temperature scaling fixes it (without touching accuracy)

Divide the logits by T and re-softmax. We sweep T, find the one minimizing ECE, and confirm the
predicted class (argmax) — hence accuracy — never changes.

In [ ]:
logit = z * sharp                                 # the raw (over-sharp) logit
Ts = np.linspace(0.5, 4, 40)
eces = [ece(sigmoid(logit / T), correct)[0] for T in Ts]
bestT = Ts[int(np.argmin(eces))]
print(f'best temperature: T = {bestT:.2f}  (≈ the true sharpness {sharp})')
print(f'ECE at T=1.0:    {ece(sigmoid(logit), correct)[0]:.3f}')
print(f'ECE at T={bestT:.2f}: {ece(sigmoid(logit/bestT), correct)[0]:.3f}  (much lower)')
# accuracy is unchanged because dividing all logits by T cannot change the argmax
print('accuracy unchanged by temperature scaling:', correct.mean())

### Validate: temperature scaling lowers ECE and preserves accuracy

Temperature scaling divides every logit by $T$. Because that is a **monotone**
transform, it cannot change which class is the argmax — so accuracy is *exactly*
preserved — yet the softened probabilities are better calibrated. We assert both:
lower ECE, identical predictions.

In [ ]:
ece_before = ece(sigmoid(logit), correct)[0]
ece_after = ece(sigmoid(logit / bestT), correct)[0]
acc_before = (sigmoid(logit) > 0.5).mean()
acc_after = (sigmoid(logit / bestT) > 0.5).mean()
print(f'ECE  T=1.0 -> {ece_before:.3f}   T={bestT:.2f} -> {ece_after:.3f}')
print(f'accuracy T=1.0 -> {acc_before:.3f}   T={bestT:.2f} -> {acc_after:.3f}')
assert ece_after < ece_before, 'temperature scaling should reduce ECE'
assert acc_before == acc_after, 'temperature scaling must not change accuracy'
# and the fitted temperature recovers the true over-sharpening factor
print(f'recovered T = {bestT:.2f}  vs true sharpness = {sharp}')
assert abs(bestT - sharp) < 0.6, 'the fitted temperature should recover the true sharpness'
print('\n✅ temperature scaling fixes calibration for free — accuracy untouched')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **ECE binning** | too few bins hides miscalibration, too many is noisy; report the scheme |
| **calibration ≠ accuracy** | fixing calibration doesn't improve labels, and vice versa (demo) |
| **temperature preserves ranking** | great — accuracy is safe — but it can't fix *which* class is wrong |
| **fit T on validation, not test** | tuning temperature on the test set leaks information |
| **distribution shift** | a temperature fit in-domain won't calibrate out-of-distribution inputs |

Demo: a model can be perfectly calibrated yet barely more accurate than a coin
flip — the axes are independent.

In [ ]:
# Accuracy and calibration are ORTHOGONAL: you can be accurate-but-miscalibrated
# (this model) or calibrated-but-inaccurate. Here we build a coin-flip-accuracy model
# that is nonetheless perfectly calibrated — proving the two axes are independent.
p = rng.uniform(0.45, 0.55, 6000)                     # honest but barely-informative probabilities
y = (rng.random(6000) < p).astype(int)                # outcome truly happens with prob p
acc = ((p > 0.5) == y).mean()
ece_cal = ece(np.where(p > 0.5, p, 1 - p), (p > 0.5) == y)[0]
print(f'this model — accuracy: {acc:.2f} (near chance)   ECE: {ece_cal:.3f} (well calibrated)')
print('\nCalibration answers "are your probabilities honest?", accuracy answers "are your')
print('labels right?" — a weather forecaster can be perfectly calibrated yet rarely certain.')

## ✏️ Your turn

**Exercise.** Implement `expected_calibration_error(conf, correct, bins)` (the weighted mean
confidence-accuracy gap) and `apply_temperature(logits, T)` (softmax over the binary case, i.e.
`sigmoid(logits / T)`). These are the measure-then-fix pair for calibration.

In [ ]:
def expected_calibration_error(conf, correct, bins=10):
    # TODO(you): bin by confidence in [0.5,1], sum (n_b/N)*|acc_b - conf_b| over non-empty bins
    return ...

def apply_temperature(logits, T):
    # TODO(you): confidence after dividing logits by temperature T (binary: sigmoid)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(expected_calibration_error(conf, correct), e0)        # matches reference
# scaling up the temperature lowers an overconfident model's ECE
assert expected_calibration_error(apply_temperature(logit, bestT), correct) < \
       expected_calibration_error(apply_temperature(logit, 1.0), correct)
# argmax (predicted class) is invariant to temperature -> accuracy preserved
assert np.array_equal(apply_temperature(logit, 1.0) > 0.5, apply_temperature(logit, 3.0) > 0.5)
print('\u2713 ECE and temperature scaling are correct')

<details>
<summary>Solution</summary>

```python
def expected_calibration_error(conf, correct, bins=10):
    edges = np.linspace(0.5, 1.0, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & ((conf < hi) | (hi == 1.0) & (conf <= hi))
        if m.sum():
            total += (m.sum()/len(conf)) * abs(correct[m].mean() - conf[m].mean())
    return total

def apply_temperature(logits, T):
    return 1 / (1 + np.exp(-logits / T))
```

Temperature scaling is the cheapest win in ML calibration: one parameter fit on validation data,
accuracy untouched (the argmax can't move), confidence made honest.

</details>

## Key takeaways

- **Accuracy ≠ calibration.** A model can be right often yet report dishonest
  probabilities; modern nets are typically **overconfident** (we confirmed it with
  sklearn's calibration curve).
- **ECE + reliability diagrams** quantify the gap between confidence and accuracy,
  bin by bin.
- **Temperature scaling is the cheap fix:** one scalar $T$ softens the logits,
  lowering ECE while leaving every prediction (and thus accuracy) unchanged — we
  verified both, and that $T$ recovered the true over-sharpening.
- **The two axes are orthogonal** — you can be calibrated-but-inaccurate or
  accurate-but-miscalibrated (demo).